## Assignment 03
Convolutional neural network (CNN)
Use MNIST Fashion Dataset and create a classifier to classify fashion clothing into categories.

In [57]:
import pandas as pd

In [58]:
train_df = pd.read_csv("fashion-mnist_train.csv")
test_df = pd.read_csv("fashion-mnist_test.csv")

In [59]:
train_df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [60]:
test_df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,0,0,0,0,0,0,0,0,9,8,...,103,87,56,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,...,34,0,0,0,0,0,0,0,0,0
2,2,0,0,0,0,0,0,14,53,99,...,0,0,0,0,63,53,31,0,0,0
3,2,0,0,0,0,0,0,0,0,0,...,137,126,140,0,133,224,222,56,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [61]:
train_df.describe()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
count,60000.000000,60000.000000,60000.000000,60000.000000,60000.000000,60000.000000,60000.000000,60000.000000,60000.000000,60000.000000,...,60000.000000,60000.000000,60000.000000,60000.000000,60000.000000,60000.000000,60000.000000,60000.000000,60000.000000,60000.00000
mean,4.500000,0.000900,0.006150,0.035333,0.101933,0.247967,0.411467,0.805767,2.198283,5.682000,...,34.625400,23.300683,16.588267,17.869433,22.814817,17.911483,8.520633,2.753300,0.855517,0.07025
std,2.872305,0.094689,0.271011,1.222324,2.452871,4.306912,5.836188,8.215169,14.093378,23.819481,...,57.545242,48.854427,41.979611,43.966032,51.830477,45.149388,29.614859,17.397652,9.356960,2.12587
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
25%,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
50%,4.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
75%,7.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,58.000000,9.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
max,9.000000,16.000000,36.000000,226.000000,164.000000,227.000000,230.000000,224.000000,255.000000,254.000000,...,255.000000,255.000000,255.000000,255.000000,255.000000,255.000000,255.000000,255.000000,255.000000,170.00000


In [62]:
train_df["label"].value_counts()

label
2    6000
9    6000
6    6000
0    6000
3    6000
4    6000
5    6000
8    6000
7    6000
1    6000
Name: count, dtype: int64

In [63]:
X_train = train_df.drop(labels=["label"], axis=1)
X_test  = test_df.drop(labels=["label"], axis=1)
y_train = train_df["label"]
y_test  = test_df["label"]

In [64]:
# Normalize the pixel values from [0, 255] to [0, 1]
X_train = X_train / 255.0
X_test = X_test / 255.0
# Labels must stay as integers 0–9; scaling them breaks one-hot encoding

In [65]:
X_train.shape

(60000, 784)

When you load images from a CSV file, the images are usually flattened out. Instead of a square grid of pixels, an image is stored as a single, long row of numbers. 

In this case, a 28 x 28 pixel image has been stretched out into a single 1D line of 784 pixels (28 x 28 = 784).

Basic neural networks (like the Dense layers you used in your previous example) actually like flat, 1D data. 

However, standard networks are terrible at recognizing images because they don't understand spatial awareness (e.g., they don't know that pixel #1 is right above pixel #29).

To do image classification properly, we will use a Convolutional Neural Network (CNN). CNNs scan images looking for shapes, edges, and corners. 

To do this, a CNN must have the data presented as a 2D grid so it knows which pixels are next to each other.
This next step simply takes that long strip of 784 pixels and folds it back up into its original 28 x 28 image shape.

In [66]:
X_train = X_train.values.reshape(-1, 28, 28, 1)
X_test = X_test.values.reshape(-1, 28, 28, 1)

You are using the NumPy .reshape() function to change the dimensions of your array without changing the actual data inside it. Here is what each number in the parentheses does:

-1 (The Wildcard / Batch Size): You have thousands of images in your dataset (let's say 60,000 in the training set). Instead of hardcoding 60000, putting -1 tells Python to do the math for you. It says: "Figure out how many total images I have and just put them all here." It ensures the code still works even if you drop a few images from your dataset later.

28, 28 (Height & Width): This tells the computer to take the flat list of 784 pixels per image and reconstruct them into a 2D grid that is 28 pixels high and 28 pixels wide.

1 (The Color Channel): Deep learning image models don't just ask for height and width; they require a "depth" dimension to understand color.

1 means there is only 1 channel: Grayscale (black and white).

If you were working with standard color images, this number would be 3 to represent the three color channels (Red, Green, Blue).

In [67]:
from tensorflow.keras.utils import to_categorical
# One-hot encoding on categorical values
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

In [68]:
# Define the model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, Dropout

model = Sequential([
    # Instead of looking at every pixel at once, it uses 32 different "filters" (each 3 x 3 pixels in size) 
    # that slide across the image. Each filter is looking for a specific feature, like a horizontal line or a sharp corner.
    # Input shape is 28x28x1 (grayscale image)
    Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    
    # This acts as a compressor. It looks at a 2 x 2 grid of pixels and only keeps the brightest (maximum) one, 
    # discarding the rest. This shrinks the image map by half, keeping the most important features while drastically 
    # reducing the math the computer has to do later.
    MaxPooling2D(2,2),

    # Second convolutional layer:
    # - 64 filters with 3x3 kernel
    # - More filters to learn more complex features
    Conv2D(64, (3,3), activation='relu'),
    # Second max pooling layer:
    MaxPooling2D(2,2),

    # At this point, the data is still in a 2D grid format (specifically, 64 tiny 2D grids). 
    # Standard neural network neurons can't process grids.
    Flatten(),

    # Dense hidden layer with 128 neurons
    # - Learns high-level patterns from the image
    Dense(128, activation='relu'),

    # Dropout layer to prevent overfitting
    # - Randomly sets 50% of neurons to 0 during training
    # This forces the remaining neurons to work harder and prevents the network from simply memorizing the training images (a problem called 'overfitting')
    Dropout(0.5),

    # Output layer:
    # - 10 neurons for 10 fashion classes
    # - Softmax activation to output probabilities that sum to 1
    Dense(10, activation='softmax')
])

c:\Users\LENOVO\Desktop\BE-Lab-Assignments\LP-V\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [69]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225,034 (879.04 KB)

 Trainable params: 225,034 (879.04 KB)

 Non-trainable params: 0 (0.00 B)

In [70]:
model.fit(
    X_train, y_train_cat,
    epochs=10,
    batch_size=64,
    validation_split=0.1
)

Epoch 1/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.7658 - loss: 0.6429 - val_accuracy: 0.8430 - val_loss: 0.4307
Epoch 2/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.8431 - loss: 0.4307 - val_accuracy: 0.8715 - val_loss: 0.3527
Epoch 3/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.8626 - loss: 0.3784 - val_accuracy: 0.8795 - val_loss: 0.3281
Epoch 4/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.8754 - loss: 0.3454 - val_accuracy: 0.8912 - val_loss: 0.2947
Epoch 5/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - accuracy: 0.8858 - loss: 0.3168 - val_accuracy: 0.9015 - val_loss: 0.2771
Epoch 6/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 12s 14ms/step - accuracy: 0.8920 - loss: 0.2952 - val_accuracy: 0.9012 - val_loss: 0.2704
Epoch 7/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - accuracy: 0.8989 - loss: 0.2786 - val_accuracy: 0.9080 - val_loss: 0.2620
Epoch 8/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - accuracy: 0.9034 - loss: 0.2604 - val_accur

In [71]:
# Run the model on test dataset and evaluate accuracy
test_loss, test_accuracy = model.evaluate(X_test, y_test_cat)
print("Test Accuracy: ", test_accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9145 - loss: 0.2285
Test Accuracy:  0.9144999980926514


In [72]:
import numpy as np
# Get predictions
y_pred = np.argmax(model.predict(X_test), axis=1)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


In [73]:
from sklearn.metrics import classification_report, confusion_matrix
print('Classification Report')
print(classification_report(y_test, y_pred))

Classification Report
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1000
           1       0.99      0.99      0.99      1000
           2       0.90      0.85      0.87      1000
           3       0.94      0.92      0.93      1000
           4       0.84      0.90      0.87      1000
           5       0.98      0.98      0.98      1000
           6       0.77      0.71      0.73      1000
           7       0.96      0.96      0.96      1000
           8       0.98      0.98      0.98      1000
           9       0.96      0.97      0.97      1000

    accuracy                           0.91     10000
   macro avg       0.91      0.91      0.91     10000
weighted avg       0.91      0.91      0.91     10000

